# 1. Imports and plotting style

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from scipy.ndimage import gaussian_filter1d
from scipy.stats import spearmanr
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "svg.fonttype": "none",        # keep text as editable text in Illustrator
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 7,
    "axes.linewidth": 0.5,
})

# 2. Paths and settings (edit here)

In [ ]:
MI_XLSX = "/directoryto/Data_S3.xlsx"
OUT_DIR = os.path.dirname(MI_XLSX)

# >>> EDIT: the AnnData objects the MI table was computed from <<<
HUMAN_H5AD = "/directoryto/Cevrim_Human_only_FibDec_cells.h5ad"
MOUSE_H5AD = "//directoryto/Cevrim_XMens_only_eFibDec_cells.h5ad"

PT_KEY      = "pseudotime"     # obs column: rank-normalized diffusion pseudotime
CT_KEY      = "cell_type"      # obs column with cell-type labels (None = use all cells)
KEEP_CT     = ["eFib", "dec"]  # cell types on the trajectory
LAYER       = "log1p_counts"     # None = .X; else name of layer holding log-normalized counts
SYMBOL_COL  = None             # None = var_names are gene symbols; else var column with symbols

N_BINS       = 100   # equal-cell-count bins along pseudotime
SMOOTH_SIGMA = 3     # Gaussian smoothing across bins (in bins)
Z_CLIP       = 3     # clip z-scores at ±3, as in the current legend

ORDER_BY        = "com"  # "com" (centre of mass, smooth diagonal) or "peak" (argmax bin)
KEEP_BLOCKS     = True   # True = early (down) block on top, then rising block
DROP_DISCORDANT = True   # drop genes whose direction disagrees between species or with their block

In [ ]:
def load_once(path):
    """Read an h5ad, keep trajectory cells, and cache what the plot needs."""
    a = ad.read_h5ad(path)
    keep = a.obs[PT_KEY].notna().to_numpy(copy=True)
    if CT_KEY is not None:
        keep &= a.obs[CT_KEY].isin(KEEP_CT).to_numpy()
    X = a.layers[LAYER] if LAYER else a.X
    X = X[keep]
    X = X.tocsc() if sp.issparse(X) else np.asarray(X)    # CSC = fast gene (column) slicing
    names = a.var[SYMBOL_COL].astype(str).values if SYMBOL_COL else a.var_names.astype(str)
    pos = pd.Series(np.arange(len(names)), index=names)
    pos = pos[~pos.index.duplicated()]                    # first match if a symbol is duplicated
    d = {"X": X, "pos": pos,
         "pt": a.obs[PT_KEY].values[keep].astype(float),
         "cells": a.obs_names[keep]}
    print(f"{os.path.basename(path)}: {keep.sum()} cells x {len(names)} genes cached")
    del a
    return d

# loads only on the first run; `del DATA` to force a reload
if "DATA" not in globals():
    DATA = {"human": load_once(HUMAN_H5AD), "mouse": load_once(MOUSE_H5AD)}

# 3. Gene list and mouse orthologs (from the MI table)

In [ ]:
# Gene panel for Fig. 5F.
# All genes are in the top 10% of pseudotime-associated genes (by MI) in both human and mouse
# Groups follow position along fibroblast-to-decidual pseudotime; within each group,
# genes are listed in order of position along human pseudotime.

PANEL = {
    "early": [                 # fibroblast genes, high early and falling
        "LAMA4",               # laminin a4, basement membrane
        "RGMA",                # repulsive guidance molecule A, BMP co-receptor
        "MATN2",               # matrilin-2, extracellular matrix
        "IGFBP5",              # IGF-binding protein 5
        "COL6A1",              # collagen VI a1, fibroblast ECM
        "DPYD",                # dihydropyrimidine dehydrogenase
        "MEIS2",               # TALE homeobox TF, HOX cofactor
        "ABI3BP",              # ABI3-binding protein, ECM
        "RORA",                # retinoic acid receptor-related orphan receptor a
        "EBF1",                # early B-cell factor 1, stromal TF
        "KCNMA1",              # BK channel a subunit
        "ROR2",                # WNT5A receptor
    ],
    "mid": [                   # genes switching around mid-pseudotime
        "PTCH1",               # hedgehog receptor, IHH target
        "PRKCE",               # protein kinase C e
        "AR",                  # androgen receptor
        "ADAMTS9",             # ADAMTS metalloprotease
        "FRS2",                # FGF receptor substrate 2
    ],
    "late": [                  # decidual genes, rising late
        "GJA1",                # connexin 43
        "FOSL2",               # AP-1 TF
        "PTTG1IP",             # PTTG1-interacting protein
        "LAMP1",               # lysosomal membrane protein
        "CD63",                # tetraspanin, lysosomal/secretory
        "TJP1",                # ZO-1, tight junctions
        "NPC2",                # lysosomal cholesterol transport
        "EPAS1",               # HIF-2a
        "CEBPB",               # C/EBPb
        "PIM1",                # PIM1 kinase
        "INHBA",               # inhibin bA (activin A)
        "BNIP3L",              # NIX, hypoxia-induced mitophagy
        "BCL2L1",              # BCL-XL, anti-apoptotic
        "SERPINE1",            # PAI-1
    ],
}

genes = pd.DataFrame([(g, grp) for grp, gs in PANEL.items() for g in gs],
                     columns=["human", "group"])
assert not genes.human.duplicated().any(), "Duplicate gene in PANEL"

mi = pd.read_excel(MI_XLSX, sheet_name="human_mouse_mi")
mi = mi.drop_duplicates("human_gene").set_index("human_gene")
missing = [g for g in genes.human if g not in mi.index]
assert not missing, f"Not in MI table: {missing}"

genes["mouse"]         = genes.human.map(mi["mouse_gene"])
genes["shared"]        = genes.human.map(mi["top_10_pct_mi"])
genes["human_mi_rank"] = genes.human.map(mi["human_mi_rank"])
genes["mouse_mi_rank"] = genes.human.map(mi["mouse_mi_rank"])

not_shared = genes.loc[genes.shared != "both", "human"].tolist()
if not_shared:
    print("WARNING, not top 10% in both species:", not_shared)
print(f"{len(genes)} genes:", genes.group.value_counts().reindex(PANEL).to_dict())
genes

# 4. Load expression along the trajectory

In [ ]:
def get_expr(d, symbols):
    """Pull (cells x genes) expression and pseudotime from the cached data (Cell 2b)."""
    absent = [g for g in symbols if g not in d["pos"].index]
    if absent:
        raise KeyError(f"genes not found: {absent}")
    X = d["X"][:, d["pos"][symbols].values]
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    return pd.DataFrame(X, index=d["cells"], columns=symbols), d["pt"]

expr_h, pt_h = get_expr(DATA["human"], genes.human.tolist())
expr_m, pt_m = get_expr(DATA["mouse"], genes.mouse.tolist())

# 5. Smooth along pseudotime and z-score per gene

In [ ]:
from scipy.interpolate import BSpline

N_KNOTS   = 5             # interior spline knots; fewer = smoother (3-6)
N_GRID    = 200           # points along pseudotime in the heatmap
MID       = (0.30, 0.70)  # pseudotime window called "mid" (0-1)
MAX_SHIFT = 0.20          # max allowed difference in position between species

def spline_z(expr, pt, n_knots=N_KNOTS, n_grid=N_GRID, clip=Z_CLIP):
    """Cubic B-spline fit of expression vs pseudotime on all cells -> per-gene z, clipped."""
    k = 3
    lo, hi = pt.min(), pt.max()
    inner = np.quantile(pt, np.linspace(0, 1, n_knots + 2)[1:-1])   # knots at cell-density quantiles
    t = np.r_[[lo] * (k + 1), inner, [hi] * (k + 1)]
    B = BSpline.design_matrix(pt, t, k).toarray()                   # cells x basis
    coef, *_ = np.linalg.lstsq(B, expr.values, rcond=None)          # fits all genes at once
    grid = np.linspace(lo, hi, n_grid)
    M = (BSpline.design_matrix(grid, t, k).toarray() @ coef).T      # genes x grid
    sd = M.std(axis=1, keepdims=True); sd[sd == 0] = 1
    Z = (M - M.mean(axis=1, keepdims=True)) / sd
    return pd.DataFrame(np.clip(Z, -clip, clip), index=expr.columns)

def com01(z):
    """Position (0-1): centre of mass of the above-mean part of a smoothed z curve."""
    x = np.linspace(0, 1, len(z)); w = np.clip(z, 0, None)
    return (w * x).sum() / w.sum() if w.sum() > 0 else x[np.argmax(z)]

Z_h = spline_z(expr_h, pt_h)
Z_m = spline_z(expr_m, pt_m)
Z_m.index = genes.human.values     # mouse rows indexed by human symbol

In [ ]:
shared = mi[mi.top_10_pct_mi == "both"]
hs = [g for g in shared.index
      if g in DATA["human"]["pos"].index and shared.loc[g, "mouse_gene"] in DATA["mouse"]["pos"].index]
eh, ph = get_expr(DATA["human"], hs)
em, pm = get_expr(DATA["mouse"], shared.loc[hs, "mouse_gene"].tolist())
Zh_all = spline_z(eh, ph)
Zm_all = spline_z(em, pm); Zm_all.index = hs

scan = pd.DataFrame({
    "mouse":         shared.loc[hs, "mouse_gene"].values,
    "pos_human":     [com01(Zh_all.loc[g].values) for g in hs],
    "pos_mouse":     [com01(Zm_all.loc[g].values) for g in hs],
    "peak_human":    Zh_all.values.argmax(axis=1) / (Zh_all.shape[1] - 1),
    "peak_mouse":    Zm_all.values.argmax(axis=1) / (Zm_all.shape[1] - 1),
    "human_mi_rank": shared.loc[hs, "human_mi_rank"].values,
    "mouse_mi_rank": shared.loc[hs, "mouse_mi_rank"].values,
}, index=pd.Index(hs, name="human"))
scan["pos_diff"] = (scan.pos_human - scan.pos_mouse).abs()
scan["stage"] = pd.cut(scan.pos_human, [0, MID[0], MID[1], 1],
                       labels=["early", "mid", "late"], include_lowest=True)

# files to send to Claude
grid = np.round(np.linspace(0, 1, Zh_all.shape[1]), 3)
scan.round(3).to_csv(os.path.join(OUT_DIR, "Fig5F_shared_gene_positions.csv"))
Zh_all.set_axis(grid, axis=1).round(3).to_csv(os.path.join(OUT_DIR, "Fig5F_all_shared_curves_human.csv"))
Zm_all.set_axis(grid, axis=1).round(3).to_csv(os.path.join(OUT_DIR, "Fig5F_all_shared_curves_mouse.csv"))

mid_hits = scan[scan.pos_human.between(*MID) & scan.pos_mouse.between(*MID)
                & (scan.pos_diff <= MAX_SHIFT)].sort_values("pos_human")
print(f"{len(scan)} shared genes scanned; {len(mid_hits)} sit mid-trajectory in both species")
print(scan.stage.value_counts().to_string())
print("\nMid genes (human symbols):\n" + " ".join(mid_hits.index))
mid_hits.round(2)

# 6. Direction check: Spearman ρ with pseudotime in each species

In [ ]:
MIN_R = 0.75   # min correlation between the human and mouse curves

genes["r_curves"]   = [np.corrcoef(Z_h.loc[g], Z_m.loc[g])[0, 1] for g in genes.human]
genes["pos_human"]  = [com01(Z_h.loc[g].values) for g in genes.human]
genes["pos_mouse"]  = [com01(Z_m.loc[g].values) for g in genes.human]
genes["pos_diff"]   = (genes.pos_human - genes.pos_mouse).abs()
genes["stage"]      = pd.cut(genes.pos_human, [0, MID[0], MID[1], 1],
                             labels=["early", "mid", "late"], include_lowest=True)
genes["concordant"] = genes["r_curves"] >= MIN_R

print("Curves differ between species (dropped)" if DROP_DISCORDANT else "Differ (kept)", ":",
      genes.loc[~genes.concordant, "human"].tolist())
print(genes.stage.value_counts().to_string())
genes.round(2)

# 7. Order rows by where each gene peaks along human pseudotime

In [ ]:
assert "pos_human" in genes, "Run Cell 6 after Cell 3 before this cell"
use = genes[genes.concordant] if DROP_DISCORDANT else genes.copy()
use = use.sort_values("pos_human")
row_h = use.human.tolist()
row_m = use.mouse.tolist()
print(f"{len(row_h)} genes plotted")
use[["human", "mouse", "stage", "pos_human", "pos_mouse", "pos_diff"]].round(2)

# 8. Plot both panels (human left, mouse right) and save as SVG

In [ ]:
CMAP = "RdBu_r"
ROW_H = 0.11            # inches per gene row
PANEL_W = 1.6           # inches per heatmap

def draw(ax, Z, labels, label_side):
    im = ax.imshow(Z, aspect="auto", cmap=CMAP, vmin=-Z_CLIP, vmax=Z_CLIP,
                   interpolation="nearest")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontstyle="italic")
    ax.set_xticks([])
    ax.tick_params(axis="y", length=0, pad=1.5)
    if label_side == "right":
        ax.yaxis.tick_right()
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xlabel("pseudotime", labelpad=2)
    return im

def add_colorbar(fig, im, rect):
    cax = fig.add_axes(rect)
    cb = fig.colorbar(im, cax=cax, ticks=[-Z_CLIP, 0, Z_CLIP])
    cb.outline.set_visible(False)
    cb.ax.tick_params(length=0, labelsize=6)
    cb.ax.set_title("z\nper gene", fontsize=6, pad=8)

H = ROW_H * len(row_h) + 0.5
Zh_plot = Z_h.loc[row_h].values
Zm_plot = Z_m.loc[row_h].values

# combined figure
fig, (axh, axm) = plt.subplots(1, 2, figsize=(2 * PANEL_W + 1.9, H),
                               gridspec_kw={"wspace": 0.08})
im = draw(axh, Zh_plot, row_h, "left");  axh.set_title("human", fontweight="bold")
draw(axm, Zm_plot, row_m, "right");      axm.set_title("mouse", fontweight="bold")
fig.subplots_adjust(left=0.17, right=0.80, top=1 - 0.3 / H, bottom=0.25 / H)
add_colorbar(fig, im, [0.93, 0.55, 0.02, 0.3])
f_both = os.path.join(OUT_DIR, "Fig5F_shared_pseudotime_genes.svg")
fig.savefig(f_both, format="svg", bbox_inches="tight")
plt.show()

# individual panels
for name, Zp, labels, side in [("human", Zh_plot, row_h, "left"),
                               ("mouse", Zm_plot, row_m, "right")]:
    fig, ax = plt.subplots(figsize=(PANEL_W + 1.2, H))
    im = draw(ax, Zp, labels, side)
    ax.set_title(name, fontweight="bold")
    if side == "left":
        fig.subplots_adjust(left=0.30, right=0.82, top=1 - 0.3 / H, bottom=0.25 / H)
        add_colorbar(fig, im, [0.88, 0.55, 0.03, 0.3])
    else:
        fig.subplots_adjust(left=0.05, right=0.62, top=1 - 0.3 / H, bottom=0.25 / H)
        add_colorbar(fig, im, [0.92, 0.55, 0.03, 0.3])
    f = os.path.join(OUT_DIR, f"Fig5F_{name}.svg")
    fig.savefig(f, format="svg", bbox_inches="tight")
    plt.close(fig)

print("Saved:", f_both, os.path.join(OUT_DIR, "Fig5F_human.svg"),
      os.path.join(OUT_DIR, "Fig5F_mouse.svg"), sep="\n  ")